In [5]:
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from osgeo import gdal
import numpy as np
import glob
from scipy.ndimage import convolve

In [ ]:
import os

os.getcwd()

'/'

In [15]:
# Step 1: Define the list of MERIT-DEM tile paths
# Replace with the actual paths to your downloaded MERIT-DEM tiles


path='/Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/data/'
dem_tiles = [
     path+'/n25w090_dem.tif',
     path+'/n25w095_dem.tif',
     path+'/n25w100_dem.tif',
     path+'/n25w105_dem.tif',
     path+'/n30w090_dem.tif',
     path+'/n30w095_dem.tif',
     path+'/n30w100_dem.tif',
     path+'/n30w105_dem.tif',
     path+'/n35w090_dem.tif',
     path+'/n35w095_dem.tif',
     path+'/n35w100_dem.tif',
     path+'/n35w105_dem.tif',
]

# Step 2: Create a Virtual Raster Tile (VRT) to mosaic the DEM tiles
vrt_path = path+'/merged_dem.vrt'
gdal.BuildVRT(vrt_path, dem_tiles)

# Step 3: Open the VRT with rasterio and check CRS
with rasterio.open(vrt_path) as dem:
    dem_crs = dem.crs
    print(f"DEM CRS: {dem_crs}")  # Check DEM CRS (should be EPSG:4326 for MERIT-DEM)


    # Read the entire DEM array into memory
    dem_array = dem.read(1)
    dem_transform = dem.transform
    dem_nodata = dem.nodata

    # Step 4: Compute slope using NumPy (no richdem)
    # Define a simple 3x3 Sobel filter for slope calculation
    dx = convolve(dem_array, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]), mode='nearest')
    dy = convolve(dem_array, np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]), mode='nearest')

    # Pixel size in meters (approximate, assuming EPSG:4326 and converting to meters)
    pixel_size_x = dem_transform[0] * 111320  # Rough meters per degree longitude at equator
    pixel_size_y = -dem_transform[4] * 111320  # Rough meters per degree latitude

    # Slope in degrees
    slope_array = np.degrees(np.arctan(np.sqrt((dx / pixel_size_x)**2 + (dy / pixel_size_y)**2)))
    slope_array[dem_array == dem_nodata] = np.nan  # Mask nodata areas

    # Step 5: Find all merged CSV files
   

    gaugepath='/Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge'

    # Step 6: Find all merged CSV files recursively
    merged_files = glob.glob(gaugepath+'/**/*_merged.csv', recursive=True)

# Check if any files were found
    if not merged_files:
        raise ValueError("No '_merged_data.csv' files found in the current directory or its subfolders.")

    # Step 6: Process each CSV file
    for file in merged_files:
        print(f"Processing {file}...")
        df = pd.read_csv(file)

        # Step 7: Check CRS of your data (assumed EPSG:4326 unless specified)
        # If your data has a different CRS, define it here (e.g., from metadata or prior knowledge)
        data_crs = 'EPSG:4326'  # Assuming WGS84 lat/lon from SWOT data
        print(f"Assumed Data CRS for {file}: {data_crs}")

        # Step 8: Reproject data coordinates to DEM CRS if they differ
        if dem_crs != data_crs:
            from pyproj import Transformer
            transformer = Transformer.from_crs(data_crs, dem_crs, always_xy=True)
            lon_transformed, lat_transformed = transformer.transform(df['longitude'].values, df['latitude'].values)
        else:
            lon_transformed = df['longitude'].values
            lat_transformed = df['latitude'].values

        elevations = []
        slopes = []

        # Step 9: Extract elevation and slope for each point
        for lon, lat in zip(lon_transformed, lat_transformed):
            try:
                # Convert lon/lat to row/col indices in the raster
                row, col = dem.index(lon, lat)
                row = int(round(row))
                col = int(round(col))

                # Check if the indices are within the raster bounds
                if 0 <= row < dem.height and 0 <= col < dem.width:
                    elev = dem_array[row, col]
                    slope_val = slope_array[row, col]
                else:
                    elev = np.nan
                    slope_val = np.nan
            except Exception as e:
                print(f"Error at point ({lon}, {lat}): {e}")
                elev = np.nan
                slope_val = np.nan

            elevations.append(elev)
            slopes.append(slope_val)

        # Step 10: Add new columns to the dataframe
        df['terrain_elevation_m'] = elevations
        df['terrain_slope_deg'] = slopes

        # Step 11: Save the updated dataframe back to the CSV
        df.to_csv(file, index=False)
        print(f"Updated {file} with terrain data.")

print("Terrain height and slope added to all merged CSV files.")

DEM CRS: EPSG:4326
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07315500_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07315500_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07315500_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07308500_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07308500_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Oklahoma/07308500_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/08028500_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Loui

/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/an

Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/07381482_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/08022500_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/08022500_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/08022500_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/07381490_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/07381490_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/07381490_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Louisiana/0736

/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/an

Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08022040_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08093100_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08093100_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08093100_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08055560_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08055560_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08055560_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08066250_merged.csv...
Assumed Data 

/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/an

Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08040600_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08374550_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08374550_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08374550_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08098290_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08098290_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08098290_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08111500_merged.csv...
Assumed Data 

/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/an

Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08159500_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08159200_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08159200_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08159200_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08088000_merged.csv...
Assumed Data CRS for /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08088000_merged.csv: EPSG:4326
Updated /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08088000_merged.csv with terrain data.
Processing /Users/adnan/Documents/Rice-Phd/Research/Geospatialdatascience/SWOT_Gauge/Texas/08090800_merged.csv...
Assumed Data 

/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:411: RuntimeWarning: invalid value encountered in cast
  new_cols = np.floor(new_cols).astype(dtype="int32")
/Users/adnan/anaconda3/envs/mapping3/lib/python3.12/site-packages/rasterio/transform.py:410: RuntimeWarning: invalid value encountered in cast
  new_rows = np.floor(new_rows).astype(dtype="int32")
/Users/adnan/an